# ThisAbled MATCH 모듈② v2 재학습 (VSCode + Colab GPU 커널)

match-input-v2 15개 특성으로 LambdaMART를 재학습한다. 로직은 저장소 `src/`·`scripts/`를
그대로 import하고(코드 복사 금지), 이 노트북은 오케스트레이션만 한다.

**실행 환경**: VSCode에서 편집하고 **커널은 Colab GPU 런타임**에 연결해 실행한다.
파일은 맥에 있고 코드는 원격 VM에서 도니, 클래식 Colab의 JS 위젯(`drive.mount`,
`files.upload`)이나 VSCode 탐색기 드래그로는 원격에 파일을 못 넣는다. 대신 **gdown**으로
개인 Drive 공유링크에서 받는다(2번 셀). Pylance의 로컬 import 경고는 무시.

**코퍼스 준비 (2번 셀 전에 1회)**:
1. 로컬 `data/processed/match_corpus.json`(약 1.4MB)을 Google Drive에 업로드
2. 우클릭 → 공유 → '링크가 있는 모든 사용자' → 링크의 `FILE_ID` 복사
3. 2번 셀의 `CORPUS_FILE_ID`에 붙여넣고 실행 → `corpus OK: 25211 ...` 확인

안 넣으면 생성기가 자체 템플릿으로 폴백한다(임베딩 현실성만 낮아지고 파이프라인은 정상 동작).

In [ ]:
# 0. 저장소 준비 (Colab 런타임에 코드 clone)
# VSCode+Colab 확장 방식: drive.mount 등 JS 위젯은 쓰지 않는다.
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/threeGuineas/thisabled-ai.git'
BRANCH = 'feature/grooming-augmentation'
REPO_DIR = Path('/content/thisabled-ai')

if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
%cd /content/thisabled-ai

In [ ]:
# 1. 의존성 설치 후 GPU 검증
!pip -q install -r requirements-colab.txt
!pip -q install pandas==2.2.3 scikit-learn==1.5.2 pyyaml==6.0.2

import torch
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 유형을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. P1.5 실문장 코퍼스 배치 (gdown — 개인 Drive 공유링크에서 받기)
# 준비:
#   1) 로컬 data/processed/match_corpus.json 을 Google Drive에 업로드
#   2) 그 파일 우클릭 → 공유 → '링크가 있는 모든 사용자' → 링크 복사
#   3) 링크 https://drive.google.com/file/d/<FILE_ID>/view... 의 <FILE_ID>를 아래에 붙여넣기
import json
from pathlib import Path

CORPUS_DST = Path('data/processed/match_corpus.json')
CORPUS_DST.parent.mkdir(parents=True, exist_ok=True)
CORPUS_FILE_ID = ''  # ← 여기에 Drive FILE_ID 붙여넣기 (예: '1AbC2dEf...')

if not CORPUS_DST.exists() and CORPUS_FILE_ID:
    import subprocess
    subprocess.run(
        ['gdown', f'https://drive.google.com/uc?id={CORPUS_FILE_ID}', '-O', str(CORPUS_DST)],
        check=True,
    )

if CORPUS_DST.exists():
    c = json.loads(CORPUS_DST.read_text(encoding='utf-8'))
    sparse = sorted(t for t, v in c.items() if len(v) < 20)
    print(f'corpus OK: {sum(len(v) for v in c.values())} sentences / {len(c)} tags')
    if sparse:
        print('  (템플릿 혼용 태그 <20:', sparse, ')')
else:
    print('코퍼스 없음(FILE_ID 미입력 또는 다운로드 실패) → 생성기가 템플릿으로 폴백.')

In [ ]:
# 3. v2 재학습 실행 (SBERT GPU 임베딩 + LambdaMART + ablation)
import sys; sys.path.insert(0, '/content/thisabled-ai')
from pathlib import Path
from scripts.train_match_v2 import run, _build_sbert

encoder = _build_sbert(Path('configs/module2_matching.yaml'))
result = run(
    encoder=encoder,
    n_users=10_000,
    n_train_queries=4_000,
    n_test_queries=1_000,
    n_candidates=20,
    seed=42,
    out_dir=Path('artifacts'),
    ablations=True,
)

In [ ]:
# 4. 지표·특성 중요도 확인
import json
print('train_pairs =', result['train_pairs'], '| test_pairs =', result['test_pairs'])
print(json.dumps(result['metrics'], ensure_ascii=False, indent=2))
print('\n[gain 중요도 top 10]')
for name, gain in list(result['gain_importance'].items())[:10]:
    print(f'  {gain:12.1f}  {name}')

# 게이트: v2_full이 legacy 임베딩 대비 동등 이상이어야 한다.
v2 = result['metrics']['v2_full']['ndcg@10']
legacy = result['metrics']['legacy_embedding']['ndcg@10']
print(f'\nNDCG@10  v2_full={v2:.4f}  legacy={legacy:.4f}  →', 'PASS' if v2 >= legacy else 'REVIEW')

In [ ]:
# 5. 산출물 위치 (VSCode 탐색기에서 다운로드)
# files.download JS 위젯 대신, artifacts/ 의 파일을 VSCode 탐색기에서 우클릭 → Download.
from pathlib import Path

for name in ('module2_lambdamart_v2.pkl', 'metrics_v2.json'):
    p = Path('artifacts') / name
    print(f'{"OK " if p.exists() else "?? "}{p.resolve()}')
print('\nVSCode 왼쪽 탐색기에서 위 파일들을 우클릭 → Download 로 로컬에 받아 P4/P5에서 검증하세요.')